# FastAPI API Test Suite

This Jupyter notebook demonstrates how to test all major endpoints in your FastAPI backend (main.py) using Python code. It covers authentication, child management, word and speaking test submissions, audio generation, and admin endpoints.

**Instructions:**
- Make sure your FastAPI server is running (default: http://localhost:8000).
- Replace sample data (emails, tokens, IDs, audio) with real values as needed.
- Cells are organized by endpoint and function for easy step-by-step testing.

In [ ]:
# 1. Import Required Libraries
import requests
import json
import base64
from pprint import pprint


In [ ]:
# 2. Set Up Test Client for FastAPI
API_URL = "http://localhost:8000"

# Helper for pretty printing JSON responses
def print_response(resp):
    try:
        pprint(resp.json())
    except Exception:
        print(resp.text)


## 3. Test User Registration and Login
Register a new user and log in to get an authentication token (`idToken`).

In [ ]:
# Register a new user (change email for each run to avoid duplicates)
register_payload = {
    "email": "testuser1@example.com",  # Change for each test run
    "name": "Test User",
    "password": "TestPassword123",
    "idToken": ""  # Not required for registration
}
resp = requests.post(f"{API_URL}/register/", json=register_payload)
print_response(resp)

# Log in with the new user
login_payload = {
    "email": register_payload["email"],
    "password": register_payload["password"]
}
resp = requests.post(f"{API_URL}/login", json=login_payload)
print_response(resp)

# Save idToken for authenticated requests
idToken = resp.json().get("id_token", "")
user_id = resp.json().get("user_id", "")


## 4. Test Add and Retrieve Child
Add a child profile and retrieve the list of children for the user.

In [ ]:
# Add a child profile
add_child_payload = {
    "idToken": idToken,
    "name": "Child One",
    "age": 7,
    "grade": "First"
}
resp = requests.post(f"{API_URL}/add_child/", json=add_child_payload)
print_response(resp)
child_id = resp.json().get("child_id", "")

# Retrieve children for the user
get_children_payload = {"idToken": idToken}
resp = requests.post(f"{API_URL}/get_children/", json=get_children_payload)
print_response(resp)


## 5. Test Submit Words and Get Results
Submit a list of words for a child and grade, then check the returned results and analysis.

In [ ]:
# Get words to test for the grade
words_payload = {"grade": "First"}
resp = requests.post(f"{API_URL}/grade/", json=words_payload)
words_list = resp.json().get("words", [])
print(f"Sample words: {[w['word'] for w in words_list[:3]]}")

# Prepare sample answers (simulate user input)
words_to_submit = [
    {"word": w["word"], "user_input": w["word"], "type": w["type"], "time": 1.0, "hints_used": 0}
    for w in words_list[:3]
]

submit_words_payload = {
    "idToken": idToken,
    "child_id": child_id,
    "grade": "First",
    "words": words_to_submit
}
resp = requests.post(f"{API_URL}/submit_words/", json=submit_words_payload)
print_response(resp)


## 6. Test Generate Audio for Words and Sentences
Call endpoints to generate audio for individual words and for all grade words, and decode base64 to verify audio.

In [ ]:
# Generate audio for a single word
word_audio_payload = {
    "idToken": idToken,
    "text": words_list[0]["word"]
}
resp = requests.post(f"{API_URL}/generate_text_audio/", json=word_audio_payload)
print_response(resp)

# Generate audio for all grade words
all_audio_payload = {"grade": "First"}
resp = requests.post(f"{API_URL}/generate_all_grade_audio/", json=all_audio_payload)
print(f"Audio files returned: {len(resp.json().get('audio_files', []))}")


## 7. Test Speaking Sentence Retrieval
Request a speaking sentence for a grade and verify the returned sentence and audio.

In [ ]:
# Request a speaking sentence for the child and grade
speaking_sentence_payload = {
    "idToken": idToken,
    "child_id": child_id,
    "grade": "First"
}
resp = requests.post(f"{API_URL}/speaking/get_sentence/", json=speaking_sentence_payload)
print_response(resp)
sentence_id = resp.json().get("sentence_id", "")
original_sentence = resp.json().get("sentence", "")


## 8. Test Speaking Submission and Analysis
Submit a speaking test (with dummy or real audio), then analyze and check the returned feedback.

In [ ]:
# NOTE: You must provide a real base64-encoded audio for a valid test.
# For demonstration, we'll use a dummy string (will fail in real API)
dummy_audio_base64 = base64.b64encode(b"dummy audio").decode()

speaking_submit_payload = {
    "idToken": idToken,
    "child_id": child_id,
    "grade": "First",
    "sentence_id": sentence_id,
    "original_sentence": original_sentence,
    "audio_base64": dummy_audio_base64,
    "audio_format": "mp3"
}
resp = requests.post(f"{API_URL}/speaking/submit/", json=speaking_submit_payload)
print_response(resp)


## 9. Test Complete Result Retrieval
Retrieve the complete result for a child and grade, and verify the summary and details.

In [ ]:
# Retrieve complete result for the child and grade
complete_result_payload = {
    "idToken": idToken,
    "child_id": child_id,
    "grade": "First"
}
resp = requests.post(f"{API_URL}/speaking/complete_result/", json=complete_result_payload)
print_response(resp)


## 10. Test Admin Endpoints (Stats, Feedback, Make Admin)
If admin credentials are available, test admin endpoints for stats, feedback retrieval, and making a user admin.

In [ ]:
# Example: Get admin stats (requires admin idToken)
# admin_idToken = "<ADMIN_ID_TOKEN>"  # Replace with a real admin token
# admin_stats_payload = {"idToken": admin_idToken}
# resp = requests.post(f"{API_URL}/admin/stats/", json=admin_stats_payload)
# print_response(resp)

# Example: Get all feedback (requires admin idToken)
# resp = requests.post(f"{API_URL}/admin/feedback/", json=admin_stats_payload)
# print_response(resp)

# Example: Make a user admin (requires admin idToken)
# make_admin_payload = {"idToken": admin_idToken, "targetEmail": "testuser1@example.com"}
# resp = requests.post(f"{API_URL}/admin/make-admin/", json=make_admin_payload)
# print_response(resp)
